In [23]:
import scanpy as sc
from anndata import AnnData
import numpy as np
import torch
from torch_geometric.data import Data
from torch_geometric.utils import coalesce, remove_self_loops
from sklearn.neighbors import NearestNeighbors
from scipy import sparse
import pandas as pd
import anndata as ad
from torch.nn import Linear
import torch.nn.functional as F

In [24]:
# adata1 = sc.read_h5ad('Chen-Stereo_seq-E15.5-s1.h5ad')
# adata2 = sc.read_h5ad('Chen-Stereo_seq-E15.5-s2.h5ad')
# adata1 = sc.read_h5ad('D:/VietBao2025/Methods/Dataset/CANCER/A73rectum_1_filter.h5ad')
# adata2 = sc.read_h5ad('D:/VietBao2025/Methods/Dataset/CANCER/A73rectum_2_filter.h5ad')
# adata1 = sc.read_h5ad('D:/VietBao2025/Methods/Dataset/Maynard/151507.h5ad')
# adata2 = sc.read_h5ad('D:/VietBao2025/Methods/Dataset/Maynard/151508.h5ad')
adata1 = sc.read_h5ad('D:/VietBao2025/Methods/Dataset/CANCER/S69_filter.h5ad') #Arutyunyan
adata2 = sc.read_h5ad('D:/VietBao2025/Methods/Dataset/CANCER/S70_filter.h5ad')
adata_train = adata1
adata_test = adata2

In [2]:
# For XENIUM

In [3]:
adata = sc.read('xenium_human_breast_cancer_analysis.h5ad')

In [4]:
adata.obs['replicates'].value_counts()

replicates
Rep_1    164000
Rep_2    118363
Name: count, dtype: int64

In [7]:
adata_train = adata[adata.obs['replicates'] == 'Rep_1'].copy()
adata_test = adata[adata.obs['replicates'] == 'Rep_2'].copy()

In [11]:
adata_train.obs.columns

Index(['cell_id', 'x_centroid', 'y_centroid', 'transcript_counts',
       'control_probe_counts', 'control_codeword_counts', 'total_counts',
       'cell_area', 'nucleus_area', 'replicates',
       ...
       'Add-on_93_GP', 'Add-on_94_GP', 'Add-on_95_GP', 'Add-on_96_GP',
       'Add-on_97_GP', 'Add-on_98_GP', 'Add-on_99_GP', 'cell_type',
       'latent_leiden_0.2', 'niche'],
      dtype='object', length=561)

In [24]:
#For Nanostring CosMX nsclc
import scanpy as sc

adata = sc.read_h5ad("downloaded_file")
# Giả sử bạn đã có:
# nsclc_adata = sc.read_h5ad(...)

# Tạo mask
train_mask = adata.obs['batch'] == 'lung5_rep1'
test_mask  = adata.obs['batch'] == 'lung6'

# Subset dữ liệu
adata_train = adata[train_mask].copy()
adata_test  = adata[test_mask].copy()

# Kiểm tra
print("Train shape:", adata_train.shape)
print("Test shape:", adata_test.shape)

Train shape: (93206, 960)
Test shape: (83723, 960)


In [25]:
train_cats = adata_train.obs["niche"].astype("category").cat.categories
test_cats  = adata_test.obs["niche"].astype("category").cat.categories

print("Train categories:", list(train_cats))
print("Test categories :", list(test_cats))

Train categories: ['immune', 'lymphoid structure', 'macrophages', 'myeloid-enriched stroma', 'neutrophils', 'plasmablast-enriched stroma', 'stroma', 'tumor interior', 'tumor-stroma boundary']
Test categories : ['immune', 'lymphoid structure', 'macrophages', 'myeloid-enriched stroma', 'stroma', 'tumor interior', 'tumor-stroma boundary']


In [26]:
# ```python
# =========================
# FULL GNN PIPELINE (5 MODELS)
# =========================

import scanpy as sc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv, GINConv, APPNP, GCNConv, GATConv
from torch_geometric.utils import add_self_loops


# =========================
# 1. FILTER COMMON LABELS
# =========================
common_labels = list(set(adata_train.obs["niche"]) & set(adata_test.obs["niche"]))

adata_train = adata_train[adata_train.obs["niche"].isin(common_labels)].copy()
adata_test  = adata_test[adata_test.obs["niche"].isin(common_labels)].copy()


# =========================
# 2. LABEL ENCODER (CHUNG)
# =========================
le = LabelEncoder()
le.fit(pd.concat([
    adata_train.obs["niche"],
    adata_test.obs["niche"]
]))


# =========================
# 3. BUILD GRAPH
# =========================
def build_graph(adata, selected_genes, k=8):
    adata = adata.copy()

    # feature
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

    X = build_X_from_genes(adata, selected_genes)

    # label
    y = torch.tensor(le.transform(adata.obs["niche"]), dtype=torch.long)

    # spatial graph
    coords = adata.obsm["spatial"]

    nbrs = NearestNeighbors(n_neighbors=k + 1)
    nbrs.fit(coords)
    _, indices = nbrs.kneighbors(coords)

    edge_list = []
    n = coords.shape[0]

    for i in range(n):
        for j in range(1, k + 1):
            nb = indices[i, j]
            edge_list.append([i, nb])
            edge_list.append([nb, i])

    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()

    edge_index, _ = add_self_loops(edge_index, num_nodes=n)

    data = Data(
        x=torch.tensor(X, dtype=torch.float32),
        edge_index=edge_index,
        y=y
    )

    return data


# =========================
# 4. MODELS
# =========================

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden)
        self.conv2 = SAGEConv(hidden, out_channels)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)


class GIN(torch.nn.Module):
    def __init__(self, in_channels, hidden, out_channels):
        super().__init__()

        nn1 = torch.nn.Sequential(
            torch.nn.Linear(in_channels, hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden, hidden)
        )

        nn2 = torch.nn.Sequential(
            torch.nn.Linear(hidden, hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden, out_channels)
        )

        self.conv1 = GINConv(nn1)
        self.conv2 = GINConv(nn2)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)


class APPNPNet(torch.nn.Module):
    def __init__(self, in_channels, hidden, out_channels):
        super().__init__()
        self.lin1 = torch.nn.Linear(in_channels, hidden)
        self.lin2 = torch.nn.Linear(hidden, out_channels)
        self.prop = APPNP(K=10, alpha=0.1)

    def forward(self, x, edge_index):
        x = F.relu(self.lin1(x))
        x = self.lin2(x)
        return self.prop(x, edge_index)


class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden)
        self.conv2 = GCNConv(hidden, out_channels)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)


class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden, out_channels, heads=4):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden, heads=heads)
        self.conv2 = GATConv(hidden * heads, out_channels, heads=1)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)


# =========================
# 5. TRAIN + EVAL + SAVE
# =========================
def train_eval_save(model, data_train, data_test, name, epochs=500):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)
    data_train = data_train.to(device)
    data_test = data_test.to(device)

    # class weights
    counts = torch.bincount(data_train.y)
    weights = 1.0 / (counts + 1e-8)
    weights = weights / weights.sum()
    weights = weights.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        out = model(data_train.x, data_train.edge_index)
        loss = loss_fn(out, data_train.y)

        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            model.eval()
            pred = model(data_test.x, data_test.edge_index).argmax(dim=1)
            acc = accuracy_score(data_test.y.cpu(), pred.cpu())
            print(f"[{name}] Epoch {epoch} | Loss {loss.item():.4f} | Acc {acc:.4f}")

    # ===== FINAL =====
    model.eval()
    pred = model(data_test.x, data_test.edge_index).argmax(dim=1)

    y_true = data_test.y.cpu().numpy()
    y_pred = pred.cpu().numpy()

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, average="weighted"),
        "precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="weighted"),
        "ARI": adjusted_rand_score(y_true, y_pred),
        "NMI": normalized_mutual_info_score(y_true, y_pred),
    }

    print(f"\n{name} metrics:")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

    # ===== SAVE CSV =====
    df = pd.DataFrame({
        "true_labels": y_true,
        "predicted_labels": y_pred
    })
    df.to_csv(f"{name.lower()}_labels.csv", index=False)

    return metrics


# =========================
# 6. RUN ALL
# =========================
selected_genes = adata_train.var_names

data_train = build_graph(adata_train, selected_genes)
data_test  = build_graph(adata_test, selected_genes)

in_channels = data_train.x.shape[1]
out_channels = len(le.classes_)
hidden = 64


models = {
    "GraphSAGE": GraphSAGE(in_channels, hidden, out_channels),
    "GIN": GIN(in_channels, hidden, out_channels),
    "APPNP": APPNPNet(in_channels, hidden, out_channels),
    "GCN": GCN(in_channels, hidden, out_channels),
    "GAT": GAT(in_channels, hidden, out_channels),
}

results = {}

for name, model in models.items():
    print(f"\n===== {name} =====")
    results[name] = train_eval_save(model, data_train, data_test, name)




===== GraphSAGE =====
[GraphSAGE] Epoch 0 | Loss 2.0872 | Acc 0.0309
[GraphSAGE] Epoch 10 | Loss 0.7186 | Acc 0.0999
[GraphSAGE] Epoch 20 | Loss 0.4850 | Acc 0.1889
[GraphSAGE] Epoch 30 | Loss 0.4019 | Acc 0.2331
[GraphSAGE] Epoch 40 | Loss 0.3576 | Acc 0.3576
[GraphSAGE] Epoch 50 | Loss 0.3278 | Acc 0.4942
[GraphSAGE] Epoch 60 | Loss 0.3053 | Acc 0.6074
[GraphSAGE] Epoch 70 | Loss 0.2863 | Acc 0.6603
[GraphSAGE] Epoch 80 | Loss 0.2693 | Acc 0.6981
[GraphSAGE] Epoch 90 | Loss 0.2557 | Acc 0.6666
[GraphSAGE] Epoch 100 | Loss 0.2442 | Acc 0.7544
[GraphSAGE] Epoch 110 | Loss 0.2354 | Acc 0.7225
[GraphSAGE] Epoch 120 | Loss 0.2254 | Acc 0.7492
[GraphSAGE] Epoch 130 | Loss 0.2165 | Acc 0.7452
[GraphSAGE] Epoch 140 | Loss 0.2085 | Acc 0.7506
[GraphSAGE] Epoch 150 | Loss 0.2009 | Acc 0.7534
[GraphSAGE] Epoch 160 | Loss 0.1937 | Acc 0.7584
[GraphSAGE] Epoch 170 | Loss 0.1902 | Acc 0.7716
[GraphSAGE] Epoch 180 | Loss 0.1830 | Acc 0.7454
[GraphSAGE] Epoch 190 | Loss 0.1765 | Acc 0.7669
[GraphSA